# Vorbereitung

- Setzen der Projektwurzel auf Workspace-Root
- Einlesen der Daten

In [ ]:
# Projektwurzel = erster Ordner nach oben mit pyproject.toml
# Pfade gelten ab Wurzel
import os
import sys
from pathlib import Path

ROOT = next(ordner for ordner in [Path.cwd(), *Path.cwd().parents] if (ordner / "pyproject.toml").exists())
os.chdir(ROOT)
# Importe aus dem Repo möglich machen
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("Projektwurzel:", ROOT)

Projektwurzel: /home/bhartl/Uni/Bachelor/BachelorArbeit/Fingerprinting-Crawler


In [ ]:
# Rohdaten einlesen: jede Zeile der JSONL-Dateien ist ein JSON-Objekt
# Konfigurationszeilen (eine je Start des Crawlers, erkennbar an path_to_output) getrennt von den Messzeilen halten
import json

DATEIEN = {
    "hist": Path("results/Forschungsfrage 1/ff1_hist.jsonl"),
    "live": Path("results/Forschungsfrage 1/ff1_live.jsonl"),
}

configs = {} # Dict mit Arrays für Live und Hist als Schlüssel
zeilen = {} # Dict mit Arrays drin für Live und Hist als Schlüssel
for name, pfad in DATEIEN.items():
    configs[name], zeilen[name] = [], []
    with open(pfad, encoding="utf-8") as datei:
        for line in datei:
            if not line.strip():
                continue
            zeile = json.loads(line)
            (configs[name] if "path_to_output" in zeile else zeilen[name]).append(zeile)
    n_websites = len({zeile["website"] for zeile in zeilen[name]})
    print(f"{name}: {len(configs[name])} Konfigurationszeilen, {len(zeilen[name])} Messzeilen, {n_websites} Websites")

hist: 3 Konfigurationszeilen, 2448 Messzeilen, 500 Websites
live: 3 Konfigurationszeilen, 729 Messzeilen, 500 Websites


# Verarbeitung

- Je Website und Stichtag nur die letzte gültige Messung
- Vollständige Websites: an allen 6 Stichtagen ein gemessener Snapshot
- Fehlerhafte Websites aus dem Panel nehmen: Fehler- oder Parkseite, Weiterleitung auf fremde Domain, leerer Titel, Serverfehler des Archivs (an beiden Messpunkten)

In [ ]:
# Je (Website, Stichtag) nur die letzte gültige Zeile behalten
# Gültig = success und internet_ok, wie beim Fortsetzen des Crawlers (core/logging.py, done_crawls)
# Kein Stichtag (None): live ist das
# normale Messung. Im Archiv sind es übersprungene Websites und Fehler vor der Snapshot-Suche, fallen aus dem Panel raus
from collections import Counter

def stichtag_von(zeile):
    # Bei Snapshots steht der Stichtag in den stats, bei Lücken und Fehlern direkt in der Zeile
    return zeile.get("stichtag") or (zeile.get("stats_vor_consent") or {}).get("stichtag")

def gueltig(zeile):
    return zeile["success"] and zeile["internet_ok"]

#Zeile ok?
def art(zeile):
    if not gueltig(zeile):
        return "fehler"
    if "stats_vor_consent" in zeile:
        return "gemessen"
    if zeile.get("no_snapshot"):
        return "kein_snapshot"
    if "skipped" in zeile:
        return "uebersprungen"
    return "unbekannt"

letzte = {}
for name in zeilen: # hist und live
    letzte[name] = {}
    # Reihenfolge der Datei = Reihenfolge der Messung: spätere Zeile ersetzt frühere
    for zeile in zeilen[name]:
        key = (zeile["website"], stichtag_von(zeile))
        bisher = letzte[name].get(key)
        # Gültige Zeile ersetzt alles, Fehlerzeile nur eine Fehlerzeile | Dict website+stichtag als key für jede komb eins
        if bisher is None or gueltig(zeile) or not gueltig(bisher):
            letzte[name][key] = zeile
    print(f"{name}: {len(zeilen[name])} Zeilen -> {len(letzte[name])} (Website, Stichtag)", dict(Counter(art(zeile) for zeile in letzte[name].values())))

hist: 2448 Zeilen -> 2264 (Website, Stichtag) {'uebersprungen': 149, 'gemessen': 1663, 'kein_snapshot': 383, 'fehler': 69}
live: 729 Zeilen -> 500 (Website, Stichtag) {'gemessen': 388, 'fehler': 112}


In [8]:
# Vollständig = an allen Stichtagen ein gemessener Snapshot
# Unvollständige bleiben erhalten (Trichter, Robustheit, Brückenpunkt), nur nicht im Panel
# Letzte Config nehmen
STICHTAGE = configs["hist"][-1]["target_dates"]
# Alle Coinfig mit der letzen Abgleichen nach stichtagen
assert all(config["target_dates"] == STICHTAGE for config in configs["hist"])

def gemessen(website, stichtag):
    zeile = letzte["hist"].get((website, stichtag))
    return zeile is not None and art(zeile) == "gemessen"

websites = sorted({website for website, _ in letzte["hist"]})
vollstaendig = [website for website in websites if all(gemessen(website, stichtag) for stichtag in STICHTAGE)]
unvollstaendig = sorted(set(websites) - set(vollstaendig))
print(f"{len(websites)} Websites: {len(vollstaendig)} vollständig, {len(unvollstaendig)} unvollständig")
# Wie viele Stichtage je Website gemessen sind (0 = übersprungen oder nie ein Snapshot)
print("Websites nach Anzahl gemessener Stichtage:", dict(sorted(Counter(sum(gemessen(website, stichtag) for stichtag in STICHTAGE) for website in websites).items())))
print("Vollständige mit Live-Messung:", sum(art(letzte["live"][(website, None)]) == "gemessen" for website in vollstaendig))

500 Websites: 201 vollständig, 299 unvollständig
Websites nach Anzahl gemessener Stichtage: {0: 171, 1: 17, 2: 18, 3: 17, 4: 27, 5: 49, 6: 201}
Vollständige mit Live-Messung: 200


In [ ]:
# Snapshot an beiden Messpunkten prüfen (Hauptmesspunkt ist nach Consent). Ein Problem an einem Stichtag nimmt die ganze Website aus dem Panel
# Fehler- oder Parkseite am Titel erkennen: Zwischenseite der Wayback Machine, Bot-Sperre, HTTP-Fehlerseite, Domain zu verkaufen oder nur reserviert
import re

FEHLERTITEL = re.compile(r"^wayback machine$|just a moment|captcha|attention required|access denied|forbidden|^404|not found|error page"
                         r"|is for sale|réservé|a bien été créé|nom de domaine déposé", re.IGNORECASE)
# Ladeseite der Wayback Machine: Snapshot zum Messzeitpunkt noch nicht geladen
LADESEITE = re.compile(r"^loading https?://web\.archive\.org/", re.IGNORECASE)

def titel_problem(titel, website):
    if not titel:
        return "leerer Titel"
    if LADESEITE.search(titel):
        return "Ladeseite Archiv"
    # HugeDomains ist ein Domain-Händler: nur Fehlerseite, wenn es nicht die Website selbst ist
    if FEHLERTITEL.search(titel):
        return "Fehler- oder Parkseite"
    return None

def probleme(website, stichtag):
    zeile = letzte["hist"][(website, stichtag)]
    gefunden = set()
    for messpunkt in ("stats_vor_consent", "stats_nach_consent"):
        stats = zeile[messpunkt]
        problem = titel_problem(stats["title"].strip(), website)
        if problem:
            gefunden.add(problem)
        if stats["cross_domain_redirect"]:
            gefunden.add("Weiterleitung fremde Domain")
        if stats["n_archive_5xx"] > 0:
            gefunden.add("Serverfehler Archiv")
    return gefunden

# Website -> Gründe über alle Stichtage
fehlerhaft = {}
for website in vollstaendig:
    gruende = set().union(*(probleme(website, stichtag) for stichtag in STICHTAGE)) 
    if gruende: # Wenn Set nicht leer dann fehlerhaft
        fehlerhaft[website] = sorted(gruende)
panel = [website for website in vollstaendig if website not in fehlerhaft]
print(f"{len(vollstaendig)} vollständig -> {len(fehlerhaft)} fehlerhaft, {len(panel)} im Panel")
print("Gründe (eine Website kann mehrere haben):", dict(Counter(grund for gruende in fehlerhaft.values() for grund in gruende)))
# Zum Prüfen von Hand: nur die Titel, die selbst als Fehler-, Park- oder Ladeseite gelten
treffer = Counter()
for website in vollstaendig:
    for stichtag in STICHTAGE:
        for punkt in ("stats_vor_consent", "stats_nach_consent"):
            titel = letzte["hist"][(website, stichtag)][punkt]["title"].strip()
            if titel_problem(titel, website) in ("Fehler- oder Parkseite", "Ladeseite Archiv"):
                treffer[titel[:80]] += 1
for titel, anzahl in treffer.most_common():
    print(f"  {anzahl:2d} x {titel}")

201 vollständig -> 91 fehlerhaft, 110 im Panel
Gründe (eine Website kann mehrere haben): {'Weiterleitung fremde Domain': 36, 'Fehler- oder Parkseite': 17, 'Serverfehler Archiv': 42, 'Ladeseite Archiv': 11, 'leerer Titel': 18}
  23 x Wayback Machine
   2 x brawlstars.com - This website is for sale! - Mma Resources and Information.
   2 x bstn.com - This website is for sale! - bstn Resources and Information.
   2 x ProDomaines : nom de domaine déposé
   2 x Le nom de domaine lwsdns.com a été réservé par le registrar www.lws.fr
   2 x Bravo ! Votre domaine lwsdns.com a bien été créé avec Hebergeur-Discount !
   2 x HugeDomains.com - MagneticMe.com is for sale (Magnetic Me)
   1 x Loading https://web.archive.org/web/20160109121437/http://arlo.com/en-us/
   1 x Loading https://web.archive.org/web/20261006210537/https://www.cdnvideo.ru/
   1 x Loading https://web.archive.org/web/20240102020827/https://www.xfinity.com/natio
   1 x Loading https://web.archive.org/web/20240101231818/https://de.

# Analyse

# Nebenfakten

- Auswahl des Panels: Verteilung auf die Rangbereiche
- Robustheit: Ergebnisse zusätzlich mit allen gültigen Snapshots statt nur dem Panel (folgt, sobald die Kennzahlen stehen)

In [ ]:
# Auswahl des Panels prüfen: Im Panel stehen nur Websites, die seit 2016 archiviert sind. Sind die Rangbereiche noch gleich stark vertreten?
# Rang aus der Tranco-Liste, Grenzen der Rangbereiche aus der Konfiguration (wie beim Ziehen der Stichprobe, utils/tranco_worker.py)
import csv
from itertools import islice

GRENZEN = configs["hist"][-1]["rank_bands"]
with open(configs["hist"][-1]["path_to_webpages"], encoding="utf-8") as datei:
    rang = {domain: int(platz) for platz, domain in islice(csv.reader(datei), GRENZEN[-1])}

def rangbereich(website):
    platz = rang[website.removeprefix("https://")]
    return next(bereich for bereich, grenze in enumerate(GRENZEN, 1) if platz <= grenze)

for name, liste in {"Stichprobe": websites, "vollständig": vollstaendig, "Panel": panel}.items():
    zaehler = Counter(rangbereich(website) for website in liste)
    print(f"{name:12s} n={len(liste):3d}  je Rangbereich:", {bereich: zaehler[bereich] for bereich in range(1, len(GRENZEN) + 1)})